In [1]:
!pip install opendatasets

In [2]:
import opendatasets as od

dataset_url = "https://www.kaggle.com/competitions/facebook-recruiting-iii-keyword-extraction"

od.download(dataset_url)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: bvishalml
Your Kaggle Key: ··········


100%|██████████| 2.90G/2.90G [01:18<00:00, 39.4MB/s]



Extracting archive ./facebook-recruiting-iii-keyword-extraction/facebook-recruiting-iii-keyword-extraction.zip to ./facebook-recruiting-iii-keyword-extraction


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import sqlite3
from nltk.corpus import stopwords

In [4]:
# Unzip Train.zip to extract Train.csv
!unzip -q /content/facebook-recruiting-iii-keyword-extraction/Train.zip -d /content/facebook-recruiting-iii-keyword-extraction/

In [ ]:
df_train=pd.read_csv("/content/facebook-recruiting-iii-keyword-extraction/Train.csv")

In [ ]:
df_train.shape

In [7]:
df_train

,Id,Title,Body,Tags
0,1,How to check if an uploaded file is an image w...,<p>I'd like to check if an uploaded file is an...,php image-processing file-upload upload mime-t...
1,2,How can I prevent firefox from closing when I ...,"<p>In my favorite editor (vim), I regularly us...",firefox
2,3,R Error Invalid type (list) for variable,<p>I am import matlab file and construct a dat...,r matlab machine-learning
3,4,How do I replace special characters in a URL?,"<p>This is probably very simple, but I simply ...",c# url encoding
4,5,How to modify whois contact details?,<pre><code>function modify(.......)\n{\n $mco...,php api file-get-contents
...,...,...,...,...
6034190,6034191,Parse JSON or XML in Status,<p>I wonder how Facebook parse the status or c...,xml facebook json status
6034191,6034192,Javascript resize on every image load,<p>I've got this code:</p>\n\n<pre><code>while...,php javascript
6034192,6034193,Update database with big CSV,<p>I need every day update ~ 10.000 items in m...,php sql query csv phpmyadmin
6034193,6034194,Difficulty adding a new view to a Roo-generate...,<p>I'm trying to add my custom view and contro...,java spring spring-mvc view spring-roo


In [8]:
df_train.isnull().sum()

,0
Id,0
Title,0
Body,0
Tags,8


In [9]:
df_train=df_train[df_train['Tags'].notnull()]

In [10]:
df_train.shape

(6034187, 4)

In [ ]:
conn = sqlite3.connect("stack.db")

chunk_size = 50000

for i in range(0, len(df_train), chunk_size):

    chunk = df_train.iloc[i:i + chunk_size]

    chunk.to_sql(
        "questions",
        conn,
        if_exists="append",
        index=False
    )

    print(f"Inserted rows: {i} to {i + len(chunk)}")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_title
ON questions(Title)
""")

conn.commit()

print("Index Created")
query = """
SELECT Title, COUNT(*) as cnt
FROM questions
GROUP BY Title
HAVING COUNT(*) > 1
ORDER BY cnt DESC
"""

duplicates = pd.read_sql_query(query, conn)
print("\nNumber of duplicate titles:",len(duplicates))

print("\nTop Duplicate Titles:\n")

print(duplicates.head(10))

query2 = """
SELECT SUM(cnt - 1) as total_duplicate_rows
FROM (
    SELECT COUNT(*) as cnt
    FROM questions
    GROUP BY Title
    HAVING COUNT(*) > 1
)
"""

total_duplicates = pd.read_sql_query(query2, conn)

print("\nTotal Duplicate Rows:")
print(total_duplicates)

conn.close()

Inserted rows: 0 to 50000
Inserted rows: 50000 to 100000
Inserted rows: 100000 to 150000
Inserted rows: 150000 to 200000
Inserted rows: 200000 to 250000
Inserted rows: 250000 to 300000
Inserted rows: 300000 to 350000
Inserted rows: 350000 to 400000
Inserted rows: 400000 to 450000
Inserted rows: 450000 to 500000
Inserted rows: 500000 to 550000
Inserted rows: 550000 to 600000
Inserted rows: 600000 to 650000
Inserted rows: 650000 to 700000
Inserted rows: 700000 to 750000
Inserted rows: 750000 to 800000
Inserted rows: 800000 to 850000
Inserted rows: 850000 to 900000
Inserted rows: 900000 to 950000
Inserted rows: 950000 to 1000000
Inserted rows: 1000000 to 1050000
Inserted rows: 1050000 to 1100000
Inserted rows: 1100000 to 1150000
Inserted rows: 1150000 to 1200000
Inserted rows: 1200000 to 1250000
Inserted rows: 1250000 to 1300000
Inserted rows: 1300000 to 1350000
Inserted rows: 1350000 to 1400000
Inserted rows: 1400000 to 1450000
Inserted rows: 1450000 to 1500000
Inserted rows: 1500000 to 

In [11]:
df_cleaned = df_train.drop_duplicates(subset=["Title"], keep="first")

print("Cleaned Shape:", df_cleaned.shape)

Cleaned Shape: (4125226, 4)


In [12]:
df_cleaned

,Id,Title,Body,Tags
0,1,How to check if an uploaded file is an image w...,<p>I'd like to check if an uploaded file is an...,php image-processing file-upload upload mime-t...
1,2,How can I prevent firefox from closing when I ...,"<p>In my favorite editor (vim), I regularly us...",firefox
2,3,R Error Invalid type (list) for variable,<p>I am import matlab file and construct a dat...,r matlab machine-learning
3,4,How do I replace special characters in a URL?,"<p>This is probably very simple, but I simply ...",c# url encoding
4,5,How to modify whois contact details?,<pre><code>function modify(.......)\n{\n $mco...,php api file-get-contents
...,...,...,...,...
6034185,6034186,Running NSTimer at 0.03seconds repeating - inc...,<pre><code> recordingTimer = [NSTimer s...,objective-c nstimer
6034186,6034187,Small Business Server 2008 Not Responding to E...,<p>I have just deployed SBS 2008 Standard and ...,windows-server-2008 dns windows-sbs-2008 small...
6034188,6034189,rdiff-backup changes permissions of source dir...,<p>I recently found rdiff-backup which seems t...,backup rsync rdiff-backup
6034192,6034193,Update database with big CSV,<p>I need every day update ~ 10.000 items in m...,php sql query csv phpmyadmin


In [19]:
df_final= df_cleaned.sample(n=10000, random_state=42)

print(df_final.shape)

(10000, 4)


In [20]:
df_final

,Id,Title,Body,Tags
5131315,5131316,pdf download option should appear,<p>I need some code to download a pdf file.</p...,php javascript download
1547058,1547059,com iplanet ias JAR,<p>I've been struggling with this for past cou...,java junit glassfish ibatis sunone
3542387,3542388,How to get the Crash Log from Apple?,"<p>We developed an app, and it works perfectly...",ios crash apple emulator appstore-approval
4065518,4065519,Android Center Buttons,<p>How do I center a button and make it so it ...,android
2997090,2997091,Parallelizing L2S Entity Retrieval,<p>Assuming a typical domain entity approach w...,sql sql-server linq-to-sql domain-driven-design
...,...,...,...,...
3307871,3307872,Snapping to specific grid positions,<p>I have a grid system laid out that lets me ...,xna
86780,86781,How to sign all the msi file which are under f...,<p>I am using signtool to sign my msi.</p>\n\n...,powershell batch-file powershell-v2.0 code-sig...
537059,537060,Cardinality of an infinite separable connected...,<p>How to prove:</p>\n\n<p>Cardinality of an i...,general-topology cardinals
2907650,2907651,What if app sleeping or dead before NSURLConne...,"<p>What will happen if an app is either dead, ...",nsurlconnection nsurlconnectiondelegate


In [21]:
df_final.to_csv("/content/df_final.csv", index=False)
print("df_final saved successfully")

df_final saved successfully
